In [90]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

In [19]:
# import train dataset
train = pd.read_csv('train.csv')
train.head()

,Unnamed: 0,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,1,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,2,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,3,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,4,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,5,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [20]:
# TRAIN:
# ignore columns: id, date and zipcode
# also ignore price column because this is the response and what we are trying to predict
columns_to_drop = ['Unnamed: 0', 'zipcode', 'price']
x_train = train.drop(columns=columns_to_drop)
x_train.head()

,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15
0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,47.5112,-122.257,1340,5650
1,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,47.7210,-122.319,1690,7639
2,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,47.7379,-122.233,2720,8062
3,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,47.5208,-122.393,1360,5000
4,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,47.6168,-122.045,1800,7503


In [21]:
# scale the data so that each feature has a mean of 0 and standard deviation of 1
scaler = StandardScaler()

# scale train dataset
for col in x_train:
    train_scaled = x_train[col].values.reshape(-1, 1) #reshape
    x_train[col] = scaler.fit_transform(train_scaled) 


# check mean and standard deviation of each feature to double check that scaling worked
for feature in x_train:
    mean = x_train.mean()
    std = x_train.var()


# organize all variables into a table
stats_train = {
    'mean': mean,
    'std': std
}


feature_data_train = pd.DataFrame(stats_train)
feature_data_train


,mean,std
bedrooms,-1.936229e-16,1.001001
bathrooms,5.329071e-17,1.001001
sqft_living,8.881784e-17,1.001001
sqft_lot,3.730349e-17,1.001001
floors,2.433609e-16,1.001001
waterfront,1.065814e-17,1.001001
view,3.463896e-17,1.001001
condition,8.526513e-17,1.001001
grade,6.394885e-17,1.001001
sqft_above,-1.048051e-16,1.001001


In [17]:
# divide the price by 1000 for all rows in the train dataset
y_train = train['price'] / 1000

In [35]:
# include a column of ones into features for the bias
# lecture slides showed column of ones in features matrix:
#shape = x_train.shape[0]
#x_train = np.append(x_train, np.ones((shape,1)), axis=1)

2. Modify your implementation from Problem 5 to implement ridge regression with gradient descent.

In [42]:
# implementation from problem 5: gradient descent for training linear regression

# add ridge regression with gradient descent
# algorithm: thetaj <- thetaj - alpha*deritivative of J(theta) 
# alpha is learning rate (small)


# features = x, price = y (both matrices?)  lam = lambda??
def gradient_descent_ridge_regression(features, price, alpha, lam, max_iterations):
    
    # initialize theta to 0?
    # length of theta vector should be amt of features + 1?
    theta = np.zeros(features.shape[1])
    N = features.shape[0]

    for iteration in range(max_iterations):
        # 1. get predicted values
        y_pred = features @ theta #?

        # 2. calc error difference btwn predicted and actual
        # y_pred = Xtheta - y 
        error = y_pred - price

        # 3. gradient descent
        # gradient = 1/N * X.T * (error calculated above) + lambda(theta)
        gradient = (1/N) * (features.T @ error) + (lam * theta)

        # 4. update:
        theta = theta - (alpha * gradient)

    return theta

In [60]:
grad_desc = gradient_descent_ridge_regression(x_train, y_train, 0.1, 1, 100)
grad_desc

array([  5.61123072,  20.55235402,  44.8877254 ,   3.99505159,
         9.85301077,  36.2654031 ,  38.46370801,  10.80477924,
        47.32051742,  35.72327681,  25.79954149, -24.07413745,
        15.95821932,  46.86976901,  -8.02628461,  42.01741312,
         2.72645104, 173.47161133, 173.47161133])

In [45]:
def predict_response(x, theta):
    return x @ theta

pred = predict_response(x_train, grad_desc)

3. Simulate $N=1000$ values of random variable $X_i$, distributed uniformly on interval $[-2,2]$. Simulate the values of random variable       $Y_i = 1 + 2X_i + e_i$, where $e_i$ is drawn from a Gaussian distribution $N(0, 2)$. Fit this data with linear regression, and also with ridge regression
for different values of $\lambda \in \{1,10,100,1000,10000\}$. Print the slope, the MSE values, and the $R^2$ statistic for each case and write down some observations. What happens as the regularization parameter $\lambda$ increases?

In [69]:
N = 1000
xi = np.random.uniform(-2, 2, N) # xi distributed uniformly on interal [-2, 2]
ei = np.random.normal(0, 2, N) # gaussian distribution
yi = 1 + 2*xi + ei
lams = [1, 10, 100, 1000, 10000] # lambda values

In [91]:
# fit data with linear regression:
xi = xi.reshape(-1, 1)

model = LinearRegression()
model.fit(xi, yi)

# print model slope
print("Slope: ", model.coef_)

Slope:  [2.07030252]


In [77]:
# print MSE and R square metrics for linear regression:
y_pred = model.predict(xi)

MSE_train = mean_squared_error(yi, y_pred)
r2_train = r2_score(yi, y_pred)

print('MSE metric: ', MSE_train)
print('R squared metric: ', r2_train)

MSE metric:  4.05322323215538
R squared metric:  0.5828142660990748


In [96]:
# fit data with ridge regression for different lambda values
mses = []
r2_scores = []

x_matrix = np.column_stack((np.ones(N), xi))

for lam in lams:

    ridge = gradient_descent_ridge_regression(xi, yi, 0.1, lam, 50)

    y_pred = predict_response(xi, ridge)

    mses.append(mean_squared_error(yi, y_pred))
    r2_scores.append(r2_score(yi, y_pred))


# organize MSEs and R squared metrics with corresponding lambdas into a table
metrics = {
    'Lambda': lams,
    'MSE metric': mses,
    'R squared metric': r2_scores
}


metrics_table = pd.DataFrame(metrics)
metrics_table


,Lambda,MSE metric,R squared metric
0,1,6.394908e+00,3.417919e-01
1,10,9.704746e+00,1.120528e-03
2,100,1.080507e+93,-1.112132e+92
3,1000,4.055288e+194,-4.173983e+193
4,10000,8.910333e+292,-9.171130e+291


As the regularization parameter, lambda, increases, the MSE metric increases as well with the most significant jump being from lambda value of 10 to lambda value of 100. Also, as the lambda increass, the R squared metric decreases with the biggest jump being from the lambda value of 10 to the lambda value of 100.